# Baseline hyperparameter selection for the paper

Selects hyperparameters for the three baselines (window Poisson GLM, GNN,
and last-available Poisson) used against the LGCP. **LGCP hyperparameter
selection has been moved out of this notebook** and is handled separately;
this notebook only covers the comparison methods.

Four evaluation weeks are used: the last two 7-day blocks of April 2025
and the last two 7-day blocks of October 2025 (each block counted
backward from the end of its month, so the blocks are contiguous and end
exactly on the last day of the month). For every method, hyperparameters
are chosen to maximize a **relative, week-averaged score**: each metric is
min-max normalized *within each week* (so 1.0 = the best combo that week,
0.0 = the worst, regardless of the metric's raw scale) before averaging
across weeks. This matters because different weeks/months can have very
different traffic levels, so a plain arithmetic average of raw metrics
would let whichever week has larger numbers dominate the choice.

**Method**

1. Last-available Poisson has no hyperparameters to search here -- it is
   fixed to its `one_step_ahead` mode (found to consistently outperform
   `frozen_train` in earlier experiments) and evaluated directly.
2. Grid-search the window Poisson GLM and the GNN, each on all four weeks,
   scoring every candidate by its relative, week-averaged
   `(mean_ll_obs, rmse_daily)`, and take the top scorer per method.
3. Assemble a final comparison table (per week + week-average) for all
   three methods.

All metrics come from `evaluate_metrics()` in `src/models/metrics_lgcp.py`.

## All hyperparameter grids

Everything that varies across this notebook's searches, in one place.

In [1]:
# --- shared ---
YEARS = [2024, 2025]
SEED = 0

# Last two 7-day blocks of April 2025, and last two 7-day blocks of
# October 2025 -- each pair counted backward from the end of the month,
# so e.g. April's blocks are 2025-04-17..23 then 2025-04-24..30 (ending
# exactly on April 30), not aligned to ISO weeks or the 1st of the month.
TEST_WEEKS = [
    {"label": "april_w1", "test_start": "2025-04-17", "test_end": "2025-04-23"},
    {"label": "april_w2", "test_start": "2025-04-24", "test_end": "2025-04-30"},
    {"label": "october_w1", "test_start": "2025-10-18", "test_end": "2025-10-24"},
    {"label": "october_w2", "test_start": "2025-10-25", "test_end": "2025-10-31"},
]

# Metrics used for the relative, week-averaged score (see methodology
# markdown cell below). "max" = higher is better, "min" = lower is better.
METRIC_DIRECTIONS = {
    "mean_ll_obs": "max",
    "rmse_daily": "min",
}

# Covariates used by every baseline (explicit override, not left to
# whatever config/14_..._glm.yaml / config/15_gnn.yaml currently say).
BASELINE_COVARIATE_COLS = [
    "chl", "thetao", "fishing_block", "is_holiday", "is_weekend",
    "cell_lag_1", "cell_lag_7",
]

# --- Last-available Poisson ---
# No longer a grid: one_step_ahead consistently outperformed frozen_train
# in earlier experiments, so it is used directly (see run below).
LAST_AVAILABLE_MODE = "one_step_ahead"

# --- Window Poisson GLM ---
GLM_ALPHA_GRID = [1e-4, 1e-3, 1e-2, 1e-1, 1.0]        # standard log-spaced L2 grid
GLM_WINDOW_SIZE_GRID = [1, 3, 7, 14]                    # daily / short / weekly / biweekly context

# --- GNN ---
GNN_HIDDEN_DIM_GRID = [16, 32, 64]
GNN_NUM_LAYERS_GRID = [1, 2, 3]
GNN_LR_GRID = [1e-4, 1e-3, 1e-2, 1e-1]                  # standard log-spaced Adam LR grid

BASE_GLM_CONFIG = "config/14_window_poisson_glm.yaml"
BASE_GNN_CONFIG = "config/15_gnn.yaml"
RESULTS_DIR_NAME = "reports/paper/baseline_hparam_apr_oct_2025"

n_glm_combos = len(TEST_WEEKS) * len(GLM_ALPHA_GRID) * len(GLM_WINDOW_SIZE_GRID)
n_gnn_combos = len(TEST_WEEKS) * len(GNN_HIDDEN_DIM_GRID) * len(GNN_NUM_LAYERS_GRID) * len(GNN_LR_GRID)
print(f"Test weeks: {[w['label'] for w in TEST_WEEKS]}")
print(f"GLM runs: {n_glm_combos}  |  GNN runs: {n_gnn_combos}  |  last-available runs: {len(TEST_WEEKS)} (fixed mode, no search)")

Test weeks: ['april_w1', 'april_w2', 'october_w1', 'october_w2']
GLM runs: 80  |  GNN runs: 144  |  last-available runs: 4 (fixed mode, no search)


In [2]:
from __future__ import annotations

import importlib.util
import itertools
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.linear_model import PoissonRegressor
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.models.metrics_lgcp import evaluate_metrics


def load_script_module(name, relpath):
    spec = importlib.util.spec_from_file_location(name, PROJECT_ROOT / relpath)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


last_avail = load_script_module("last_avail", "scripts/13_evaluate_last_available_poisson.py")
window_glm = load_script_module("window_glm", "scripts/14_train_window_poisson_glm.py")
gnn_mod = load_script_module("gnn_mod", "scripts/15_train_gnn.py")

RESULTS_DIR = PROJECT_ROOT / RESULTS_DIR_NAME
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Modules loaded OK. Results will be saved under: {RESULTS_DIR}")

Modules loaded OK. Results will be saved under: /home/rdoz/PhD/AIPS/reports/paper/baseline_hparam_apr_oct_2025


## Methodology helper

**`compute_relative_scores`**: for each metric, min-max normalizes across
the hyperparameter combos *within each week* (1.0 = best that week), then
averages across weeks per combo. A combo is only kept if it converged
(produced valid results) on every week. `combined_relative_score` is the
mean of all relative metric columns and is what every grid search in this
notebook ranks by.

In [3]:
METRIC_KEYS = [
    "mean_ll_obs", "mae_obs", "rmse_obs",
    "mean_ll_daily", "mae_daily", "rmse_daily",
    "daily_delta_corr", "daily_direction_accuracy_moving",
]


def compute_relative_scores(df: pd.DataFrame, week_col: str, group_cols: list[str], metrics_directions: dict) -> pd.DataFrame:
    group_cols = list(group_cols)
    valid_df = df.dropna(subset=list(metrics_directions)).copy()

    for metric, direction in metrics_directions.items():
        def _normalize(s, direction=direction):
            lo, hi = s.min(), s.max()
            if hi - lo < 1e-12:
                return pd.Series(1.0, index=s.index)
            return (s - lo) / (hi - lo) if direction == "max" else (hi - s) / (hi - lo)
        valid_df[f"relative_{metric}"] = valid_df.groupby(week_col)[metric].transform(_normalize)

    relative_cols = [f"relative_{m}" for m in metrics_directions]
    n_weeks_total = df[week_col].nunique()
    week_counts = valid_df.groupby(group_cols)[week_col].nunique().rename("n_weeks_valid")

    scores = valid_df.groupby(group_cols)[relative_cols].mean()
    scores["combined_relative_score"] = scores[relative_cols].mean(axis=1)
    scores = scores.join(week_counts).reset_index()

    complete = scores[scores["n_weeks_valid"] == n_weeks_total].copy()
    incomplete = scores[scores["n_weeks_valid"] != n_weeks_total]
    if len(incomplete):
        print(f"  Excluded {len(incomplete)} combo(s) that didn't converge on every week:")
        display(incomplete[group_cols + ["n_weeks_valid"]])
    return complete.sort_values("combined_relative_score", ascending=False)


print("Helpers defined.")

Helpers defined.


## Step 1 — Last-available Poisson (fixed mode, no search)

In [4]:
la_cfg = last_avail.load_config(None)
la_cfg["years"] = YEARS
la_cfg["parquet_path"] = str(PROJECT_ROOT / la_cfg["parquet_path"])

la_rows = []
for week in TEST_WEEKS:
    label, test_start, test_end = week["label"], week["test_start"], week["test_end"]

    (_, _, _, _, _, test_y, _, df_la) = last_avail.prepare_data(
        la_cfg["parquet_path"], train_fraction=la_cfg["train_fraction"], random_seed=la_cfg["data_seed"],
        split_strategy="fixed_test_window", test_start_date=test_start, test_end_date=test_end,
        years=YEARS,
    )
    df_la["date"] = pd.to_datetime(df_la["date"])
    train_mask, test_mask = last_avail.make_day_split_masks(
        df=df_la, train_fraction=la_cfg["train_fraction"], random_seed=la_cfg["data_seed"],
        split_strategy="fixed_test_window", test_start_date=test_start, test_end_date=test_end,
    )
    test_dates = df_la.loc[test_mask, "date"].values
    preds = last_avail.build_last_available_predictions(
        df=df_la, train_mask=train_mask, test_mask=test_mask, mode=LAST_AVAILABLE_MODE
    )
    metrics = evaluate_metrics(test_y, preds, test_dates)
    print(f"[week={label}] mean_ll_obs={metrics['mean_ll_obs']:.4f}  rmse_daily={metrics['rmse_daily']:.4f}")
    la_rows.append({"week": label, **metrics})

la_results_df = pd.DataFrame(la_rows)
la_results_df.to_csv(RESULTS_DIR / "last_available_results.csv", index=False)
display(la_results_df)

[week=april_w1] mean_ll_obs=-2.3123  rmse_daily=9.5169
[week=april_w2] mean_ll_obs=-4.2570  rmse_daily=15.6159


[week=october_w1] mean_ll_obs=-3.2358  rmse_daily=16.3532


[week=october_w2] mean_ll_obs=-3.8776  rmse_daily=11.5820


,week,mean_ll_obs,mae_obs,rmse_obs,mean_ll_daily,mae_daily,rmse_daily,daily_delta_corr,daily_direction_accuracy_moving
0,april_w1,-2.312258,0.163265,0.716324,-29.053399,5.428571,9.516902,-0.263474,0.2
1,april_w2,-4.257028,0.486880,1.429591,-31.649820,11.571429,15.615926,0.074097,0.5
2,october_w1,-3.235776,0.367347,1.010153,-109.045449,13.428571,16.353243,-0.242573,0.2
3,october_w2,-3.877622,0.393586,0.988269,-103.972849,7.857143,11.582006,-0.087234,0.4


## Step 2 — Window Poisson GLM grid search

`poisson_alpha` (L2 regularization) over a standard log-spaced grid, x
`window_size` (autoregressive context length) over daily/short/weekly/
biweekly (1/3/7/14 days).

In [5]:
glm_cfg = window_glm.load_config(str(PROJECT_ROOT / BASE_GLM_CONFIG))
glm_cfg["years"] = YEARS
glm_cfg["parquet_path"] = str(PROJECT_ROOT / glm_cfg["parquet_path"])
glm_cfg["covariate_cols"] = BASELINE_COVARIATE_COLS

df_glm_raw = window_glm.load_parquet_years(glm_cfg["parquet_path"], YEARS)
df_glm_raw["date"] = pd.to_datetime(df_glm_raw["date"])

glm_rows = []
for week in TEST_WEEKS:
    label, test_start, test_end = week["label"], week["test_start"], week["test_end"]
    for window_size in GLM_WINDOW_SIZE_GRID:
        cfg_ws = dict(glm_cfg)
        cfg_ws["window_size"] = window_size
        df_win, feature_cols = window_glm.build_window_dataframe(df_glm_raw, cfg_ws)
        train_mask, test_mask = window_glm.make_split_masks(
            dates=df_win["date"], split_strategy="fixed_test_window",
            train_fraction=cfg_ws["train_fraction"], random_seed=cfg_ws["data_seed"],
            test_start_date=test_start, test_end_date=test_end,
        )
        X = df_win[feature_cols].values.astype(np.float32)
        y = df_win[cfg_ws["target_col"]].values.astype(np.float32)
        dts = df_win["date"].values
        X_train, y_train = X[train_mask], y[train_mask]
        X_test, y_test = X[test_mask], y[test_mask]
        test_dates = dts[test_mask]

        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_test_s = scaler.transform(X_test)

        for alpha in GLM_ALPHA_GRID:
            model = PoissonRegressor(alpha=alpha, max_iter=cfg_ws["max_iter"])
            model.fit(X_train_s, y_train)
            preds = np.clip(model.predict(X_test_s), 0, None)
            metrics = evaluate_metrics(y_test, preds, test_dates)
            glm_rows.append({"week": label, "window_size": window_size, "poisson_alpha": alpha, **metrics})
    print(f"[week={label}] GLM grid done.")

glm_results_df = pd.DataFrame(glm_rows)
glm_results_df.to_csv(RESULTS_DIR / "glm_grid_results.csv", index=False)
glm_scores_df = compute_relative_scores(glm_results_df, week_col="week", group_cols=["window_size", "poisson_alpha"], metrics_directions=METRIC_DIRECTIONS)
display(glm_scores_df.head(10))
best_glm_row = glm_scores_df.iloc[0]
BEST_GLM_WINDOW_SIZE = int(best_glm_row["window_size"])
BEST_GLM_ALPHA = float(best_glm_row["poisson_alpha"])
print(f"Selected GLM config: window_size={BEST_GLM_WINDOW_SIZE}  poisson_alpha={BEST_GLM_ALPHA}")

[week=april_w1] GLM grid done.


[week=april_w2] GLM grid done.


[week=october_w1] GLM grid done.


[week=october_w2] GLM grid done.


,window_size,poisson_alpha,relative_mean_ll_obs,relative_rmse_daily,combined_relative_score,n_weeks_valid
17,14,0.0100,0.977446,0.813567,0.895506,4
16,14,0.0010,0.969592,0.815006,0.892299,4
15,14,0.0001,0.966115,0.814204,0.890160,4
11,7,0.0010,0.779073,0.861459,0.820266,4
10,7,0.0001,0.776254,0.861950,0.819102,4
12,7,0.0100,0.779654,0.849820,0.814737,4
18,14,0.1000,0.877331,0.679994,0.778663,4
6,3,0.0010,0.626557,0.812039,0.719298,4
5,3,0.0001,0.624704,0.813591,0.719148,4
7,3,0.0100,0.622988,0.792353,0.707670,4


Selected GLM config: window_size=14  poisson_alpha=0.01


## Step 3 — GNN grid search

`hidden_dim` and `num_layers` over typical small-GCN capacities, x `lr`
over a standard log-spaced Adam range.

In [6]:
gnn_cfg = gnn_mod.load_config(str(PROJECT_ROOT / BASE_GNN_CONFIG))
gnn_cfg["years"] = YEARS
gnn_cfg["parquet_path"] = str(PROJECT_ROOT / gnn_cfg["parquet_path"])
gnn_cfg["covariate_cols"] = BASELINE_COVARIATE_COLS
device = torch.device(gnn_cfg.get("device", "cpu"))

df_gnn = gnn_mod.load_parquet_years(gnn_cfg["parquet_path"], YEARS)
df_gnn["date"] = pd.to_datetime(df_gnn["date"])
df_feat, feature_cols = gnn_mod.build_node_features(df_gnn, gnn_cfg)
X, Y, dates, cell_coords = gnn_mod.build_graph_tensors(df_feat, feature_cols, gnn_cfg["target_col"])
n_dates, n_cells, n_features = X.shape

adj = gnn_mod.build_grid_adjacency(
    cell_coords[["longitude", "latitude"]].to_numpy(), connectivity=gnn_cfg.get("adjacency", "queen")
)
adj_norm = gnn_mod.normalize_adjacency(adj)
adj_norm_t = torch.tensor(adj_norm, dtype=torch.float32, device=device)

gnn_rows = []
for week in TEST_WEEKS:
    label, test_start, test_end = week["label"], week["test_start"], week["test_end"]

    train_mask, test_mask = gnn_mod.make_day_split_masks(
        df=pd.DataFrame({"date": dates}), train_fraction=gnn_cfg["train_fraction"], random_seed=gnn_cfg["data_seed"],
        split_strategy="fixed_test_window", test_start_date=test_start, test_end_date=test_end,
    )
    X_train, Y_train = X[train_mask], Y[train_mask]
    X_test, Y_test = X[test_mask], Y[test_mask]
    dates_test = dates[test_mask]

    scaler = StandardScaler()
    scaler.fit(X_train.reshape(-1, n_features))
    X_train_model = scaler.transform(X_train.reshape(-1, n_features)).reshape(X_train.shape)
    X_test_model = scaler.transform(X_test.reshape(-1, n_features)).reshape(X_test.shape)

    X_train_t = torch.tensor(X_train_model, dtype=torch.float32, device=device)
    Y_train_t = torch.tensor(Y_train, dtype=torch.float32, device=device)
    X_test_t = torch.tensor(X_test_model, dtype=torch.float32, device=device)

    for hidden_dim, num_layers, lr in itertools.product(GNN_HIDDEN_DIM_GRID, GNN_NUM_LAYERS_GRID, GNN_LR_GRID):
        torch.manual_seed(SEED)
        model = gnn_mod.SpatioTemporalGNN(
            in_dim=n_features, hidden_dim=hidden_dim, num_layers=num_layers, dropout=gnn_cfg["dropout"],
        ).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=gnn_cfg["weight_decay"])

        model.train()
        for epoch in range(1, int(gnn_cfg["num_epochs"]) + 1):
            optimizer.zero_grad()
            rate_train = model(X_train_t, adj_norm_t)
            loss = gnn_mod.poisson_nll(rate_train, Y_train_t)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            rate_test = model(X_test_t, adj_norm_t).cpu().numpy()
        rate_test = np.clip(rate_test, 0, None)

        y_test_flat = Y_test.reshape(-1)
        rate_test_flat = rate_test.reshape(-1)
        test_dates_flat = np.repeat(dates_test, n_cells)
        metrics = evaluate_metrics(y_test_flat, rate_test_flat, test_dates_flat)
        gnn_rows.append({"week": label, "hidden_dim": hidden_dim, "num_layers": num_layers, "lr": lr, **metrics})
    print(f"[week={label}] GNN grid done.")

gnn_results_df = pd.DataFrame(gnn_rows)
gnn_results_df.to_csv(RESULTS_DIR / "gnn_grid_results.csv", index=False)
gnn_scores_df = compute_relative_scores(gnn_results_df, week_col="week", group_cols=["hidden_dim", "num_layers", "lr"], metrics_directions=METRIC_DIRECTIONS)
display(gnn_scores_df.head(10))
best_gnn_row = gnn_scores_df.iloc[0]
BEST_GNN_HIDDEN_DIM = int(best_gnn_row["hidden_dim"])
BEST_GNN_NUM_LAYERS = int(best_gnn_row["num_layers"])
BEST_GNN_LR = float(best_gnn_row["lr"])
print(f"Selected GNN config: hidden_dim={BEST_GNN_HIDDEN_DIM}  num_layers={BEST_GNN_NUM_LAYERS}  lr={BEST_GNN_LR}")

Generated lag features: ['cell_lag_1', 'cell_lag_7', 'cell_roll_mean_7', 'daily_total_lag_1', 'daily_total_lag_7', 'daily_total_roll_mean_7']


[week=april_w1] GNN grid done.


[week=april_w2] GNN grid done.


[week=october_w1] GNN grid done.


[week=october_w2] GNN grid done.


,hidden_dim,num_layers,lr,relative_mean_ll_obs,relative_rmse_daily,combined_relative_score,n_weeks_valid
34,64,3,0.01,0.902617,0.943992,0.923304,4
30,64,2,0.01,0.948213,0.808199,0.878206,4
27,64,1,0.10,0.923400,0.827721,0.875560,4
23,32,3,0.10,0.920447,0.826829,0.873638,4
18,32,2,0.01,0.910423,0.822106,0.866264,4
22,32,3,0.01,0.898824,0.833478,0.866151,4
15,32,1,0.10,0.922780,0.806710,0.864745,4
35,64,3,0.10,0.770219,0.917944,0.844082,4
31,64,2,0.10,0.922707,0.743278,0.832992,4
19,32,2,0.10,0.883598,0.757211,0.820405,4


Selected GNN config: hidden_dim=64  num_layers=3  lr=0.01


## Final comparison table

In [7]:
baseline_rows = []
for row in la_results_df.to_dict("records"):
    baseline_rows.append({"method": "Last Available Poisson", "week": row["week"], **{k: row[k] for k in METRIC_KEYS}})
for row in glm_results_df[
    (glm_results_df["window_size"] == BEST_GLM_WINDOW_SIZE) & (glm_results_df["poisson_alpha"] == BEST_GLM_ALPHA)
].to_dict("records"):
    baseline_rows.append({"method": "Window Poisson GLM", "week": row["week"], **{k: row[k] for k in METRIC_KEYS}})
for row in gnn_results_df[
    (gnn_results_df["hidden_dim"] == BEST_GNN_HIDDEN_DIM)
    & (gnn_results_df["num_layers"] == BEST_GNN_NUM_LAYERS)
    & (gnn_results_df["lr"] == BEST_GNN_LR)
].to_dict("records"):
    baseline_rows.append({"method": "GNN", "week": row["week"], **{k: row[k] for k in METRIC_KEYS}})

final_by_week_df = pd.DataFrame(baseline_rows)
final_by_week_df.to_csv(RESULTS_DIR / "final_comparison_by_week.csv", index=False)

hyperparameters = {
    "Last Available Poisson": f"mode={LAST_AVAILABLE_MODE}",
    "Window Poisson GLM": f"window_size={BEST_GLM_WINDOW_SIZE}, poisson_alpha={BEST_GLM_ALPHA}",
    "GNN": f"hidden_dim={BEST_GNN_HIDDEN_DIM}, num_layers={BEST_GNN_NUM_LAYERS}, lr={BEST_GNN_LR}",
}

final_avg_df = final_by_week_df.groupby("method")[METRIC_KEYS].mean()
final_avg_df.insert(0, "hyperparameters", final_avg_df.index.map(hyperparameters))
final_avg_df = final_avg_df.loc[["Last Available Poisson", "Window Poisson GLM", "GNN"]]
final_avg_df.to_csv(RESULTS_DIR / "final_comparison_avg.csv")

print("Per-week detail:")
display(final_by_week_df.set_index(["method", "week"]))

print(f"\nWeek-averaged final comparison (weeks: {[w['label'] for w in TEST_WEEKS]}):")
print(f"All intermediate results + this table saved under: {RESULTS_DIR}\n")
final_avg_df

Per-week detail:


mean_ll_obs   mae_obs  rmse_obs  \
method                 week                                          
Last Available Poisson april_w1      -2.312258  0.163265  0.716324   
                       april_w2      -4.257028  0.486880  1.429591   
                       october_w1    -3.235776  0.367347  1.010153   
                       october_w2    -3.877622  0.393586  0.988269   
Window Poisson GLM     april_w1      -0.441460  0.256482  0.763105   
                       april_w2      -0.798497  0.444760  1.295397   
                       october_w1    -0.612115  0.418662  0.914063   
                       october_w2    -0.609320  0.399341  0.743302   
GNN                    april_w1      -0.363674  0.214153  0.679449   
                       april_w2      -0.692414  0.356027  1.019536   
                       october_w1    -0.490879  0.318799  0.811653   
                       october_w2    -0.567262  0.319254  0.725385   

                                   mean_ll_daily  mae_daily  rmse_daily  \
method                 week                                               
Last Available Poisson april_w1       -29.053399   5.428571    9.516902   
                       april_w2       -31.649820  11.571429   15.615926   
                       october_w1    -109.045449  13.428571   16.353243   
                       october_w2    -103.972849   7.857143   11.582006   
Window Poisson GLM     april_w1        -4.977197   6.161286    7.708395   
                       april_w2        -6.871081   9.511999   12.512510   
                       october_w1      -6.314919   8.658906   10.371515   
                       october_w2      -3.198091   4.457552    5.582672   
GNN                    april_w1        -4.310913   3.774387    4.472775   
                       april_w2        -4.755299   6.032465    9.329484   
                       october_w1      -5.491056   8.130772   10.223643   
                       october_w2      -3.534374   3.755664    5.248322   

                                   daily_delta_corr  \
method                 week                           
Last Available Poisson april_w1           -0.263474   
                       april_w2            0.074097   
                       october_w1         -0.242573   
                       october_w2         -0.087234   
Window Poisson GLM     april_w1            0.730058   
                       april_w2            0.575978   
                       october_w1          0.340904   
                       october_w2          0.901243   
GNN                    april_w1            0.755711   
                       april_w2            0.717423   
                       october_w1          0.388163   
                       october_w2          0.753031   

                                   daily_direction_accuracy_moving  
method                 week                                         
Last Available Poisson april_w1                                0.2  
                       april_w2                                0.5  
                       october_w1                              0.2  
                       october_w2                              0.4  
Window Poisson GLM     april_w1                                0.6  
                       april_w2                                0.5  
                       october_w1                              0.4  
                       october_w2                              1.0  
GNN                    april_w1                                0.4  
                       april_w2                                0.5  
                       october_w1                              0.6  
                       october_w2                              0.8


Week-averaged final comparison (weeks: ['april_w1', 'april_w2', 'october_w1', 'october_w2']):
All intermediate results + this table saved under: /home/rdoz/PhD/AIPS/reports/paper/baseline_hparam_apr_oct_2025



,hyperparameters,mean_ll_obs,mae_obs,rmse_obs,mean_ll_daily,mae_daily,rmse_daily,daily_delta_corr,daily_direction_accuracy_moving
method,,,,,,,,,
Last Available Poisson,mode=one_step_ahead,-3.420671,0.352770,1.036084,-68.430379,9.571429,13.267019,-0.129796,0.325
Window Poisson GLM,"window_size=14, poisson_alpha=0.01",-0.615348,0.379811,0.928967,-5.340322,7.197436,9.043773,0.637046,0.625
GNN,"hidden_dim=64, num_layers=3, lr=0.01",-0.528557,0.302058,0.809006,-4.522910,5.423322,7.318556,0.653582,0.575


**Notes for the paper writeup**

- All 3 methods are evaluated on the identical four test weeks (last two
  7-day blocks of April 2025, last two of October 2025) and trained on the
  identical `years=[2024, 2025]` data available before each week.
- Last-available Poisson uses its `one_step_ahead` mode directly (no
  search); the GLM and GNN hyperparameters were chosen by grid search over
  all four weeks, scoring by the relative (min-max normalized per week)
  average of `(mean_ll_obs, rmse_daily)`.
- `final_comparison_by_week.csv` has the per-week breakdown;
  `final_comparison_avg.csv` is the headline week-averaged table above.
- LGCP hyperparameter selection is handled in a separate notebook; once
  finalized, its results should use the same four test weeks as here for
  a consistent final comparison.